In [1]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

C:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\


Failed to read module file 'C:\Users\Admin\AppData\Roaming\uv\python\cpython-3.12.8-windows-x86_64-none\Lib\pydoc_data\topics.py' for module 'pydoc_data.topics': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Admin\AppData\Roaming\uv\python\cpython-3.12.8-windows-x86_64-none\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<

In [2]:
import pandas as pd

import torch
from models.resnet import ResnetMultilabel
from models.mobilenet import MobileNetMultilabel
from models.quant_mobilenet import load_mobilenet_v3_quant

from training.cross_validation import run_cross_val, train_model


## Running the Optimization Experiments

This section covers the model optimization experiments:
- Switching from **ResNet18** to **MobileNet V3 Small**,
- Further **Truncating** the MobileNet architecture,
- Applying **8-bit Quantization-Aware Training (QAT)** on the MobileNet model.

Each experiment is executed with **cross-validation** as described in the paper.  
Results are written to a dedicated `results/` directory and subsequently examined in the `results_analysis` folder.



In [15]:
labels_df = pd.read_csv("../data/Verified_Dataset/labels/labels_merged.csv")
labels_df["ClipFilenamePt"] = labels_df["clip_filename"].str.replace(".wav", ".pt", regex=False)


label_columns = ["ECHO", "HFPC", "BBPC", "Whistle"]

data_dir = "../data"
processed_spects_dir = data_dir + "/Verified_Dataset/spectrograms/"

# labels_df =labels_df[labels_df["Boat"] != 1]
labels_df =labels_df[labels_df["labeling_effort"].isin(["old_abs_from_3s_snippets", "old_abs_CAC_last_samples", "irene_20min", "irene_5min"])]

labels_df.reset_index(drop=True, inplace=True)

results_dir = "./results/new_dataset"

In [16]:
labels_df[labels_df["HFPC"].isna()]

,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),...,clip_start_time,clip_end_time,DETAILS,boat_labeling_file,boat_labeling_file_id,annotator,SnippetFilename,start_s,end_s,ClipFilenamePt


In [17]:
training_config_default = {
    "batch_size": 32,
    "lr_decay_factor": 0.5,
    "patience_lr": 2,
    # "n_epochs": 1, #100
    # "min_epochs": 0, #15
    "n_epochs": 100, #100
    "min_epochs": 10, #15
    "patience_early_stopping": 5,
    "metric_mode": "max",
    "val_metric": "f1",
}

In [18]:
import torch
torch.cuda.is_available()

True

### Resnet18


In [19]:
run_cross_val(
    labels_df, 
    label_columns, 
    ResnetMultilabel,  
    processed_spects_dir,
    run_name="resnet_only_irene",
    results_dir=results_dir,
    model_kwargs={
        "pretrained":True,
    }, 
    training_config=training_config_default,
    save_models=True,
    use_quantization=False,
)


Val Epoch: 20


100%|██████████| 42/42 [00:03<00:00, 12.33it/s]


Val Epoch: 20 Results - 
loss: 9.618, 
accuracy: {'Labels_Average': 0.9836219549179077, 'ECHO': 0.9804216623306274, 'HFPC': 0.9871987700462341, 'BBPC': 0.9849397540092468, 'Whistle': 0.9819276928901672}, 
f1: {'Labels_Average': 0.901005744934082, 'ECHO': 0.8879310488700867, 'HFPC': 0.882758617401123, 'BBPC': 0.9047619104385376, 'Whistle': 0.9285714030265808}, 
precision: {'Labels_Average': 0.9237242937088013, 'ECHO': 0.8728813529014587, 'HFPC': 0.9014084339141846, 'BBPC': 0.9693877696990967, 'Whistle': 0.9512194991111755}, 
recall: {'Labels_Average': 0.8808912038803101, 'ECHO': 0.9035087823867798, 'HFPC': 0.8648648858070374, 'BBPC': 0.8482142686843872, 'Whistle': 0.9069767594337463}, 
AUC: {'Labels_Average': 0.9961872100830078, 'ECHO': 0.9949095249176025, 'HFPC': 0.996605396270752, 'BBPC': 0.9970262050628662, 'Whistle': 0.9962078332901001}, 
exact_match: {'Labels_Average': 0.946536123752594},

Training Epoch: 21


100%|██████████| 166/166 [00:20<00:00,  8.10it/s]


Train Epoch: 21 Results - 
loss: 0.463, 
accuracy: {'Labels_Average': 0.9990110993385315, 'ECHO': 0.999058187007904, 'HFPC': 0.9992465376853943, 'BBPC': 0.9992465376853943, 'Whistle': 0.9984931349754333}, 
f1: {'Labels_Average': 0.9949530363082886, 'ECHO': 0.995112419128418, 'HFPC': 0.9943820238113403, 'BBPC': 0.9953810572624207, 'Whistle': 0.9949367046356201}, 
precision: {'Labels_Average': 0.9966465830802917, 'ECHO': 0.998039186000824, 'HFPC': 0.997183084487915, 'BBPC': 0.9976851940155029, 'Whistle': 0.993678867816925}, 
recall: {'Labels_Average': 0.9932711720466614, 'ECHO': 0.9922027587890625, 'HFPC': 0.9915966391563416, 'BBPC': 0.9930875301361084, 'Whistle': 0.9961977005004883}, 
AUC: {'Labels_Average': 0.999971866607666, 'ECHO': 0.9999910593032837, 'HFPC': 0.9999406337738037, 'BBPC': 0.9999645352363586, 'Whistle': 0.9999910593032837}, 
exact_match: {'Labels_Average': 0.996232807636261},

Val Epoch: 21


...

100%|██████████| 40/40 [00:04<00:00,  9.84it/s]


Test on site bsm Epoch: 0 Results - 
loss: 3.770, 
accuracy: {'Labels_Average': 0.9924842119216919, 'ECHO': 0.9905063509941101, 'HFPC': 0.9912974834442139, 'BBPC': 0.9944620132446289, 'Whistle': 0.9936708807945251}, 
f1: {'Labels_Average': 0.8849285244941711, 'ECHO': 0.8947368264198303, 'HFPC': 0.8253968358039856, 'BBPC': 0.892307698726654, 'Whistle': 0.9272727370262146}, 
precision: {'Labels_Average': 0.892875611782074, 'ECHO': 0.8947368264198303, 'HFPC': 0.8965517282485962, 'BBPC': 0.8529411554336548, 'Whistle': 0.9272727370262146}, 
recall: {'Labels_Average': 0.8805497884750366, 'ECHO': 0.8947368264198303, 'HFPC': 0.7647058963775635, 'BBPC': 0.9354838728904724, 'Whistle': 0.9272727370262146}, 
AUC: {'Labels_Average': 0.9978962540626526, 'ECHO': 0.9972020387649536, 'HFPC': 0.9977522492408752, 'BBPC': 0.9981948137283325, 'Whistle': 0.998435914516449}, 
exact_match: {'Labels_Average': 0.9723101258277893},
                   Filename Site  ECHO_true  ECHO_pred    ECHO_probs  \
0  BSM_20

100%|██████████| 5/5 [00:00<00:00, 11.60it/s]


Test on site rdl Epoch: 0 Results - 
loss: 17.382, 
accuracy: {'Labels_Average': 0.9534883499145508, 'ECHO': 0.930232584476471, 'HFPC': 0.9689922332763672, 'BBPC': 0.9379844665527344, 'Whistle': 0.9767441749572754}, 
f1: {'Labels_Average': 0.897951602935791, 'ECHO': 0.8363636136054993, 'HFPC': 0.9230769276618958, 'BBPC': 0.8620689511299133, 'Whistle': 0.9702970385551453}, 
precision: {'Labels_Average': 0.8995758295059204, 'ECHO': 0.8518518805503845, 'HFPC': 0.9599999785423279, 'BBPC': 0.8064516186714172, 'Whistle': 0.9800000190734863}, 
recall: {'Labels_Average': 0.8992569446563721, 'ECHO': 0.8214285969734192, 'HFPC': 0.8888888955116272, 'BBPC': 0.9259259104728699, 'Whistle': 0.9607843160629272}, 
AUC: {'Labels_Average': 0.9818958044052124, 'ECHO': 0.9653465747833252, 'HFPC': 0.9901960492134094, 'BBPC': 0.9753086566925049, 'Whistle': 0.9967319965362549}, 
exact_match: {'Labels_Average': 0.8372092843055725},
                   Filename Site  ECHO_true  ECHO_pred  ECHO_probs  HFPC_true  

,Filename,Site,ECHO_true,ECHO_pred,ECHO_probs,HFPC_true,HFPC_pred,HFPC_probs,BBPC_true,BBPC_pred,BBPC_probs,Whistle_true,Whistle_pred,Whistle_probs
0,RDL_20200722_10495100.pt,RDL,0.0,1.0,6.653182e-01,0.0,0.0,2.541073e-07,1.0,1.0,1.000000e+00,1.0,1.0,9.999983e-01
1,RDL_20200722_10530700.pt,RDL,0.0,0.0,1.494951e-06,0.0,0.0,6.434662e-09,1.0,1.0,1.000000e+00,1.0,1.0,9.998509e-01
2,RDL_20200722_10530800.pt,RDL,0.0,0.0,3.102565e-06,0.0,0.0,5.433516e-08,1.0,1.0,9.999998e-01,1.0,1.0,9.999675e-01
3,RDL_20200722_10535400.pt,RDL,0.0,0.0,3.364059e-05,0.0,0.0,2.661448e-06,0.0,0.0,8.817448e-04,0.0,0.0,2.467944e-03
4,RDL_20200722_10560000.pt,RDL,0.0,0.0,4.447617e-01,0.0,0.0,1.353020e-06,1.0,1.0,9.999996e-01,1.0,1.0,9.999981e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
639,RDL_20200828_18471143.pt,RDL,1.0,1.0,9.999815e-01,0.0,0.0,2.397118e-04,1.0,1.0,6.450806e-01,1.0,1.0,9.994324e-01
640,RDL_20200828_18482301.pt,RDL,0.0,0.0,2.559880e-02,1.0,1.0,7.016237e-01,1.0,1.0,6.422930e-01,0.0,0.0,1.629033e-01
641,RDL_20200829_13033300.pt,RDL,0.0,0.0,3.636325e-03,0.0,0.0,2.510605e-05,1.0,1.0,5.081753e-01,1.0,1.0,9.971179e-01
642,RDL_20200906_14553196.pt,RDL,0.0,0.0,7.232674e-08,0.0,0.0,9.718217e-11,0.0,0.0,5.850543e-12,0.0,0.0,3.262010e-13


#### Running all MobileNet variants: layer depth & quantization 

In [6]:

n_layers_to_test = [8, 6,  10, 12,]  
# n_layers_to_test = [2, 4, 6, 8, 10, 12,]  
quantization_options = [False,True]

for n_layers in n_layers_to_test:
    for use_quantization in quantization_options:
        # Create run name based on parameters
        quant_suffix = "_qat" if use_quantization else ""
        run_name = f"mobile_net{quant_suffix}_{n_layers}_layers"
        
        print(f"\n{'='*80}")
        print(f"Running experiment: {run_name}")
        print(f"n_layers: {n_layers}, quantization: {use_quantization}")
        print(f"{'='*80}")

        model_class = load_mobilenet_v3_quant if use_quantization else MobileNetMultilabel

        model_kwargs = {
            "pretrained": True,
            "n_layers": n_layers
        }

        if use_quantization:
            model_kwargs["qat"] = True
        
        try:
            run_cross_val(
                labels_df, 
                label_columns, 
                model_class,  
                processed_spects_dir,
                run_name=run_name,
                model_kwargs=model_kwargs, 
                n_splits=5,
                training_config=training_config_default,
                save_models=True,
                use_quantization=use_quantization,
            )
            print(f"✅ Successfully completed: {run_name}")
            
        except Exception as e:
            print(f"❌ Error in experiment {run_name}: {str(e)}")
            print(f"Continuing with next experiment...")
            continue

print(f"\n{'='*80}")
print("All experiments completed!")
print(f"{'='*80}")


Val Epoch: 25


100%|██████████| 78/78 [00:09<00:00,  7.86it/s]


Val Epoch: 25 Results - 
loss: 18.293, 
accuracy: {'Labels_Average': 0.9726137518882751, 'ECHO': 0.9629480242729187, 'HFPC': 0.9855014085769653, 'BBPC': 0.9810712933540344, 'Whistle': 0.9609343409538269}, 
f1: {'Labels_Average': 0.8788601756095886, 'ECHO': 0.9471871256828308, 'HFPC': 0.843478262424469, 'BBPC': 0.8303248882293701, 'Whistle': 0.8944504857063293}, 
precision: {'Labels_Average': 0.8883719444274902, 'ECHO': 0.9482758641242981, 'HFPC': 0.843478262424469, 'BBPC': 0.8778625726699829, 'Whistle': 0.8838709592819214}, 
recall: {'Labels_Average': 0.8706341981887817, 'ECHO': 0.9461008906364441, 'HFPC': 0.843478262424469, 'BBPC': 0.7876712083816528, 'Whistle': 0.9052863717079163}, 
AUC: {'Labels_Average': 0.9874508380889893, 'ECHO': 0.9902142286300659, 'HFPC': 0.9879093766212463, 'BBPC': 0.9862003326416016, 'Whistle': 0.9854792356491089}, 
exact_match: {'Labels_Average': 0.9065646529197693},

Training Epoch: 26


100%|██████████| 311/311 [00:46<00:00,  6.72it/s]


Train Epoch: 26 Results - 
loss: 0.456, 
accuracy: {'Labels_Average': 0.9988672733306885, 'ECHO': 0.9984897375106812, 'HFPC': 0.9989931583404541, 'BBPC': 0.9997986555099487, 'Whistle': 0.9981876611709595}, 
f1: {'Labels_Average': 0.9953732490539551, 'ECHO': 0.997732400894165, 'HFPC': 0.990138053894043, 'BBPC': 0.9984301328659058, 'Whistle': 0.995192289352417}, 
precision: {'Labels_Average': 0.9951527118682861, 'ECHO': 0.9978832602500916, 'HFPC': 0.990138053894043, 'BBPC': 0.9968652129173279, 'Whistle': 0.9957242012023926}, 
recall: {'Labels_Average': 0.9955951571464539, 'ECHO': 0.9975816011428833, 'HFPC': 0.990138053894043, 'BBPC': 1.0, 'Whistle': 0.9946609735488892}, 
AUC: {'Labels_Average': 0.9999841451644897, 'ECHO': 0.9999843835830688, 'HFPC': 0.9999895691871643, 'BBPC': 0.9999959468841553, 'Whistle': 0.9999667406082153}, 
exact_match: {'Labels_Average': 0.9954692125320435},

Val Epoch: 26


...

(quantized) test on site rdl Epoch: 0 Results - 
loss: 51.184, 
accuracy: {'Labels_Average': 0.9166666865348816, 'ECHO': 0.9069767594337463, 'HFPC': 0.930232584476471, 'BBPC': 0.8914728760719299, 'Whistle': 0.9379844665527344}, 
f1: {'Labels_Average': 0.862175703048706, 'ECHO': 0.8666666746139526, 'HFPC': 0.8831169009208679, 'BBPC': 0.7586206793785095, 'Whistle': 0.9402984976768494}, 
precision: {'Labels_Average': 0.8604111671447754, 'ECHO': 0.8863636255264282, 'HFPC': 0.8292682766914368, 'BBPC': 0.7857142686843872, 'Whistle': 0.9402984976768494}, 
recall: {'Labels_Average': 0.8664755821228027, 'ECHO': 0.8478260636329651, 'HFPC': 0.9444444179534912, 'BBPC': 0.7333333492279053, 'Whistle': 0.9402984976768494}, 
AUC: {'Labels_Average': 0.9691227674484253, 'ECHO': 0.9566526412963867, 'HFPC': 0.9734169840812683, 'BBPC': 0.9661616086959839, 'Whistle': 0.9802598357200623}, 
exact_match: {'Labels_Average': 0.7441860437393188},
Created test dataloader for site: CAC with 657 samples

Test on sit

100%|██████████| 21/21 [00:02<00:00,  7.92it/s]


Test on site cac Epoch: 0 Results - 
loss: 36.316, 
accuracy: {'Labels_Average': 0.9406392574310303, 'ECHO': 0.9330289363861084, 'HFPC': 0.9726027250289917, 'BBPC': 0.95281583070755, 'Whistle': 0.9041095972061157}, 
f1: {'Labels_Average': 0.8473471403121948, 'ECHO': 0.9153845906257629, 'HFPC': 0.7804877758026123, 'BBPC': 0.7891156673431396, 'Whistle': 0.9044005870819092}, 
precision: {'Labels_Average': 0.8712427020072937, 'ECHO': 0.9370078444480896, 'HFPC': 0.8421052694320679, 'BBPC': 0.8055555820465088, 'Whistle': 0.9003021121025085}, 
recall: {'Labels_Average': 0.8259698748588562, 'ECHO': 0.8947368264198303, 'HFPC': 0.7272727489471436, 'BBPC': 0.7733333110809326, 'Whistle': 0.9085366129875183}, 
AUC: {'Labels_Average': 0.9623892307281494, 'ECHO': 0.971083402633667, 'HFPC': 0.9613674283027649, 'BBPC': 0.9525887370109558, 'Whistle': 0.96451735496521}, 
exact_match: {'Labels_Average': 0.8036529421806335},
                   Filename Site  ECHO_true  ECHO_pred  ECHO_probs  HFPC_true  \
0

100%|██████████| 21/21 [00:02<00:00,  7.19it/s]


(quantized) test on site cac Epoch: 0 Results - 
loss: 40.183, 
accuracy: {'Labels_Average': 0.9379755854606628, 'ECHO': 0.9254185557365417, 'HFPC': 0.9726027250289917, 'BBPC': 0.9558599591255188, 'Whistle': 0.8980212807655334}, 
f1: {'Labels_Average': 0.8471972942352295, 'ECHO': 0.9052224159240723, 'HFPC': 0.7804877758026123, 'BBPC': 0.8053691387176514, 'Whistle': 0.8977099061012268}, 
precision: {'Labels_Average': 0.8710674047470093, 'ECHO': 0.9322709441184998, 'HFPC': 0.8421052694320679, 'BBPC': 0.8108108043670654, 'Whistle': 0.8990825414657593}, 
recall: {'Labels_Average': 0.8258283138275146, 'ECHO': 0.8796992301940918, 'HFPC': 0.7272727489471436, 'BBPC': 0.800000011920929, 'Whistle': 0.8963414430618286}, 
AUC: {'Labels_Average': 0.9596536159515381, 'ECHO': 0.9710882306098938, 'HFPC': 0.9504486918449402, 'BBPC': 0.9534821510314941, 'Whistle': 0.9635953903198242}, 
exact_match: {'Labels_Average': 0.7960426211357117},
✅ Successfully completed: mobile_net_qat_12_layers

All experiment

<h4>Function call template to run quick experiments<h4>


In [ ]:
run_cross_val(
    labels_df, 
    label_columns, 
    MobileNetMultilabel,  
    processed_spects_dir,
    run_name="mobile_net_hp_1024_8_layers_all_absences",
    model_kwargs={
        "pretrained":True,
        "n_layers": 8
    }, 
    n_splits=5,
    training_config=training_config_default,
    save_models=True,
    use_quantization=False,
)

## Site Generalization Experiments


1. **Site-specific models** — train a separate model per site.
2. **Leave-One-Site-Out** — train on all but one site and test on the held-out site to assess generalizability.

All runs follow the protocol described in the paper.


In [20]:
from training.cross_validation import create_test_fold_indices
from sklearn.model_selection import KFold, train_test_split
from models.utils import aggregate_folds_testing_metrics



labels_df = pd.read_csv("../data/Verified_Dataset/labels/labels_merged.csv")
labels_df["ClipFilenamePt"] = labels_df["clip_filename"].str.replace(".wav", ".pt", regex=False)


label_columns = ["ECHO", "HFPC", "BBPC", "Whistle"]

data_dir = "../data"
processed_spects_dir = data_dir + "/Verified_Dataset/spectrograms/"

results_dir = "./results/new_dataset"
# labels_df =labels_df[labels_df["labeling_effort"].isin([ "irene_20min", "irene_5min"])]
# labels_df =labels_df[labels_df["labeling_effort"].isin(["old_abs_from_3s_snippets", "old_abs_CAC_last_samples", "irene_20min", "irene_5min"])]
labels_df =labels_df[labels_df["Site"] != "BSM"]
labels_df.reset_index(drop=True, inplace=True)

labels_df = create_test_fold_indices(labels_df, 5)

In [22]:
labels_df["Call"].value_counts()

Call
1    5475
0    2161
Name: count, dtype: int64

In [8]:
labels_df.columns

Index(['clip_filename', 'ECHO', 'BBPC', 'HFPC', 'Whistle', 'Call', 'Boat',
       'labeling_effort', 'DETAIL', 'Begin Time (s)', 'End Time (s)',
       'GROUNDTRUTH', 'Notes', 'original_filename', 'labeled_snippet_filename',
       'snippet_start_time', 'snippet_start_s', 'snippet_end_s', 'Site',
       'labeled_snippet_dir', 'HydrophoneModel', 'HydrophoneSensitivity',
       'clip_start_time', 'clip_end_time', 'DETAILS', 'boat_labeling_file',
       'boat_labeling_file_id', 'annotator', 'SnippetFilename', 'start_s',
       'end_s', 'ClipFilenamePt'],
      dtype='str')

### Site-specific models

In [13]:
all_sites = ["RDL", "CAC", "BSM", "KAM" ]

use_quantization = False

for train_site in all_sites:

    for fold_idx in range(5):
        
        model_class = load_mobilenet_v3_quant if use_quantization else MobileNetMultilabel

        model = model_class(
            pretrained=True,
            n_layers=8,
            num_classes=len(label_columns)
        )

        train_site_df = labels_df[labels_df["Site"]==train_site]
        train_data = train_site_df[train_site_df["test_fold_idx"] != fold_idx]
        train_data, val_data = train_test_split(train_data, test_size=0.2, random_state=42, stratify=train_data['Site'])
        
        test_data = train_site_df[train_site_df["test_fold_idx"] == fold_idx]

        run_name = f"{train_site}_only"
        if use_quantization:
            run_name = run_name + "_qat"


        run_dir, _, _ = train_model(
            labels_df,
            label_columns,
            model,
            train_data,
            val_data,
            test_data,
            processed_spects_dir=processed_spects_dir,
            fold_idx=fold_idx,
            results_dir="./results/sites_generalization_no_boat",
            run_name=run_name,
            training_config=training_config_default,
            use_quantization=use_quantization,
            compute_sites_metrics=True
        )

    aggregate_folds_testing_metrics(run_dir)


Val Epoch: 40


100%|██████████| 8/8 [00:00<00:00, 12.46it/s]


Val Epoch: 40 Results - 
loss: 22.455, 
accuracy: {'Labels_Average': 0.9597457647323608, 'ECHO': 0.9788135886192322, 'HFPC': 0.9788135886192322, 'BBPC': 0.9661017060279846, 'Whistle': 0.9152542352676392}, 
f1: {'Labels_Average': 0.8806450366973877, 'ECHO': 0.9854227304458618, 'HFPC': 0.761904776096344, 'BBPC': 0.8888888955116272, 'Whistle': 0.8863636255264282}, 
precision: {'Labels_Average': 0.9094022512435913, 'ECHO': 0.9768785834312439, 'HFPC': 0.8888888955116272, 'BBPC': 0.8648648858070374, 'Whistle': 0.9069767594337463}, 
recall: {'Labels_Average': 0.8604341745376587, 'ECHO': 0.9941176176071167, 'HFPC': 0.6666666865348816, 'BBPC': 0.9142857193946838, 'Whistle': 0.8666666746139526}, 
AUC: {'Labels_Average': 0.9536234140396118, 'ECHO': 0.9959001541137695, 'HFPC': 0.8623511791229248, 'BBPC': 0.9945985078811646, 'Whistle': 0.9616438746452332}, 
exact_match: {'Labels_Average': 0.8516949415206909},

Training Epoch: 41


100%|██████████| 30/30 [00:03<00:00,  9.68it/s]


Train Epoch: 41 Results - 
loss: 0.486, 
accuracy: {'Labels_Average': 0.9989373087882996, 'ECHO': 1.0, 'HFPC': 1.0, 'BBPC': 0.9989373087882996, 'Whistle': 0.9968119263648987}, 
f1: {'Labels_Average': 0.9979671835899353, 'ECHO': 1.0, 'HFPC': 1.0, 'BBPC': 0.9961977005004883, 'Whistle': 0.9956709742546082}, 
precision: {'Labels_Average': 0.9985590577125549, 'ECHO': 1.0, 'HFPC': 1.0, 'BBPC': 1.0, 'Whistle': 0.9942362904548645}, 
recall: {'Labels_Average': 0.9973835349082947, 'ECHO': 1.0, 'HFPC': 1.0, 'BBPC': 0.9924242496490479, 'Whistle': 0.9971098303794861}, 
AUC: {'Labels_Average': 0.9999915361404419, 'ECHO': 1.0, 'HFPC': 1.0, 'BBPC': 1.0, 'Whistle': 0.999966025352478}, 
exact_match: {'Labels_Average': 0.9957491755485535},

Val Epoch: 41


...

Test on site rdl Epoch: 0 Results - 
loss: 54.142, 
accuracy: {'Labels_Average': 0.8841463327407837, 'ECHO': 0.707317054271698, 'HFPC': 0.9674796462059021, 'BBPC': 0.9024389982223511, 'Whistle': 0.9593495726585388}, 
f1: {'Labels_Average': 0.8175476789474487, 'ECHO': 0.5909090638160706, 'HFPC': 0.8999999761581421, 'BBPC': 0.8235294222831726, 'Whistle': 0.9557521939277649}, 
precision: {'Labels_Average': 0.7498221397399902, 'ECHO': 0.4333333373069763, 'HFPC': 0.8571428656578064, 'BBPC': 0.7777777910232544, 'Whistle': 0.931034505367279}, 
recall: {'Labels_Average': 0.9331895112991333, 'ECHO': 0.9285714030265808, 'HFPC': 0.9473684430122375, 'BBPC': 0.875, 'Whistle': 0.9818181991577148}, 
AUC: {'Labels_Average': 0.9533776640892029, 'ECHO': 0.8774435520172119, 'HFPC': 0.9984817504882812, 'BBPC': 0.9469436407089233, 'Whistle': 0.990641713142395}, 
exact_match: {'Labels_Average': 0.5853658318519592},
                   Filename Site  ECHO_true  ECHO_pred  ECHO_probs  HFPC_true  \
0  RDL_20200

100%|██████████| 14/14 [00:01<00:00,  9.98it/s]


Test on site cac Epoch: 0 Results - 
loss: 57.053, 
accuracy: {'Labels_Average': 0.8987414240837097, 'ECHO': 0.9038901329040527, 'HFPC': 0.9656750559806824, 'BBPC': 0.8993135094642639, 'Whistle': 0.8260869383811951}, 
f1: {'Labels_Average': 0.7573743462562561, 'ECHO': 0.8703703880310059, 'HFPC': 0.6666666865348816, 'BBPC': 0.6507936716079712, 'Whistle': 0.8416666388511658}, 
precision: {'Labels_Average': 0.8437986969947815, 'ECHO': 0.892405092716217, 'HFPC': 0.9375, 'BBPC': 0.5694444179534912, 'Whistle': 0.9758453965187073}, 
recall: {'Labels_Average': 0.7164562344551086, 'ECHO': 0.849397599697113, 'HFPC': 0.517241358757019, 'BBPC': 0.7592592835426331, 'Whistle': 0.7399267554283142}, 
AUC: {'Labels_Average': 0.9282768964767456, 'ECHO': 0.963144063949585, 'HFPC': 0.9023833870887756, 'BBPC': 0.9201478958129883, 'Whistle': 0.9274322986602783}, 
exact_match: {'Labels_Average': 0.6453089118003845},
                   Filename Site  ECHO_true  ECHO_pred  ECHO_probs  HFPC_true  \
0  CAC_20210

100%|██████████| 44/44 [00:04<00:00, 10.07it/s]

Test on site bsm Epoch: 0 Results - 
loss: 23.437, 
accuracy: {'Labels_Average': 0.9517416954040527, 'ECHO': 0.958635687828064, 'HFPC': 0.9477503895759583, 'BBPC': 0.9600870609283447, 'Whistle': 0.9404934644699097}, 
f1: {'Labels_Average': 0.6215034127235413, 'ECHO': 0.8308605551719666, 'HFPC': 0.5324675440788269, 'BBPC': 0.4954128563404083, 'Whistle': 0.6272727251052856}, 
precision: {'Labels_Average': 0.4951186180114746, 'ECHO': 0.7216494679450989, 'HFPC': 0.40594059228897095, 'BBPC': 0.3802816867828369, 'Whistle': 0.4726027250289917}, 
recall: {'Labels_Average': 0.8488911390304565, 'ECHO': 0.9790209531784058, 'HFPC': 0.7735849022865295, 'BBPC': 0.7105262875556946, 'Whistle': 0.9324324131011963}, 
AUC: {'Labels_Average': 0.9732560515403748, 'ECHO': 0.994269609451294, 'HFPC': 0.9560270309448242, 'BBPC': 0.958375871181488, 'Whistle': 0.9843516945838928}, 
exact_match: {'Labels_Average': 0.8613933324813843},
                   Filename Site  ECHO_true  ECHO_pred  ECHO_probs  HFPC_true  

### Leave One Site out

In [24]:


# all_sites = ["BSM", "RDL", "CAC", "KAM" ]
all_sites = [ "RDL", "CAC", "KAM" ]

use_quantization = False

for out_site in all_sites:
    for fold_idx in range(5):
        
        model_class = load_mobilenet_v3_quant if use_quantization else MobileNetMultilabel

        model = model_class(
            pretrained=True,
            n_layers=8,
            num_classes=len(label_columns)
        )

        train_sites = [site for site in all_sites if site != out_site]

        train_df = labels_df[labels_df["test_fold_idx"] != fold_idx]
        
        train_sites_df = train_df[train_df["Site"].isin(train_sites)]

        train_data, val_data = train_test_split(train_sites_df, test_size=0.2, random_state=42, stratify=train_sites_df['Site'])
        
        test_data = labels_df[labels_df["test_fold_idx"] == fold_idx]


        run_name = f"leave_{out_site}_out"
        if use_quantization:
            run_name = run_name + "_qat"

        print(run_name)
        print(f"Site out : {out_site}")
        print("Train df")
        print(train_data["Site"].value_counts())
        print("\nVal df")
        print(val_data["Site"].value_counts())

        # break
        run_dir, _, _ = train_model(
            labels_df,
            label_columns,
            model,
            train_data,
            val_data,
            test_data,
            processed_spects_dir=processed_spects_dir,
            fold_idx=fold_idx,
            run_name=run_name,
            results_dir="./results/sites_generalization_no_bsm",
            training_config=training_config_default,
            use_quantization=use_quantization,
            
        )
    
    aggregate_folds_testing_metrics(run_dir)

    # break


Val Epoch: 25


100%|██████████| 20/20 [00:01<00:00, 14.73it/s]


Val Epoch: 25 Results - 
loss: 25.708, 
accuracy: {'Labels_Average': 0.952305257320404, 'ECHO': 0.9554848670959473, 'HFPC': 0.9825119376182556, 'BBPC': 0.9650238752365112, 'Whistle': 0.9062002897262573}, 
f1: {'Labels_Average': 0.89054936170578, 'ECHO': 0.9385964870452881, 'HFPC': 0.904347836971283, 'BBPC': 0.8225806355476379, 'Whistle': 0.8966724872589111}, 
precision: {'Labels_Average': 0.8815526366233826, 'ECHO': 0.9385964870452881, 'HFPC': 0.8524590134620667, 'BBPC': 0.8644067645072937, 'Whistle': 0.8707482814788818}, 
recall: {'Labels_Average': 0.9025906324386597, 'ECHO': 0.9385964870452881, 'HFPC': 0.9629629850387573, 'BBPC': 0.7846153974533081, 'Whistle': 0.9241877198219299}, 
AUC: {'Labels_Average': 0.9816229343414307, 'ECHO': 0.9861584901809692, 'HFPC': 0.9958454370498657, 'BBPC': 0.9739225506782532, 'Whistle': 0.9705653786659241}, 
exact_match: {'Labels_Average': 0.8314785361289978},

Training Epoch: 26


100%|██████████| 79/79 [00:07<00:00, 11.23it/s]


Train Epoch: 26 Results - 
loss: 1.098, 
accuracy: {'Labels_Average': 0.997714638710022, 'ECHO': 0.995230495929718, 'HFPC': 0.9988076090812683, 'BBPC': 0.9980127215385437, 'Whistle': 0.9988076090812683}, 
f1: {'Labels_Average': 0.9942843317985535, 'ECHO': 0.9937499761581421, 'HFPC': 0.9928740859031677, 'BBPC': 0.9917080998420715, 'Whistle': 0.9988052845001221}, 
precision: {'Labels_Average': 0.9948363900184631, 'ECHO': 0.9937499761581421, 'HFPC': 0.9905213117599487, 'BBPC': 0.996666669845581, 'Whistle': 0.9984076619148254}, 
recall: {'Labels_Average': 0.9937474727630615, 'ECHO': 0.9937499761581421, 'HFPC': 0.9952380657196045, 'BBPC': 0.9867987036705017, 'Whistle': 0.9992032051086426}, 
AUC: {'Labels_Average': 0.9999399185180664, 'ECHO': 0.999882161617279, 'HFPC': 0.9998987913131714, 'BBPC': 0.9999850988388062, 'Whistle': 0.999993622303009}, 
exact_match: {'Labels_Average': 0.9908584952354431},

Val Epoch: 26


...

Test on site cac Epoch: 0 Results - 
loss: 28.945, 
accuracy: {'Labels_Average': 0.9505327343940735, 'ECHO': 0.9406392574310303, 'HFPC': 0.9847792983055115, 'BBPC': 0.9665144681930542, 'Whistle': 0.9101978540420532}, 
f1: {'Labels_Average': 0.8761004209518433, 'ECHO': 0.9239766001701355, 'HFPC': 0.84375, 'BBPC': 0.8253968358039856, 'Whistle': 0.9112781882286072}, 
precision: {'Labels_Average': 0.9098043441772461, 'ECHO': 0.9595141410827637, 'HFPC': 0.8999999761581421, 'BBPC': 0.8387096524238586, 'Whistle': 0.9409937858581543}, 
recall: {'Labels_Average': 0.8452442288398743, 'ECHO': 0.8909774422645569, 'HFPC': 0.7941176295280457, 'BBPC': 0.8125, 'Whistle': 0.8833819031715393}, 
AUC: {'Labels_Average': 0.9744966626167297, 'ECHO': 0.9805299043655396, 'HFPC': 0.9796053171157837, 'BBPC': 0.9735982418060303, 'Whistle': 0.9642531871795654}, 
exact_match: {'Labels_Average': 0.8219178318977356},
                   Filename Site  ECHO_true  ECHO_pred  ECHO_probs  HFPC_true  \
0  CAC_20210714_080

100%|██████████| 24/24 [00:01<00:00, 13.00it/s]


Test on site kam Epoch: 0 Results - 
loss: 88.941, 
accuracy: {'Labels_Average': 0.8090418577194214, 'ECHO': 0.8475033640861511, 'HFPC': 0.9757084846496582, 'BBPC': 0.7206477522850037, 'Whistle': 0.692307710647583}, 
f1: {'Labels_Average': 0.6114455461502075, 'ECHO': 0.8871129155158997, 'HFPC': 0.6666666865348816, 'BBPC': 0.36307692527770996, 'Whistle': 0.5289255976676941}, 
precision: {'Labels_Average': 0.5844823718070984, 'ECHO': 0.9758241772651672, 'HFPC': 0.75, 'BBPC': 0.23228345811367035, 'Whistle': 0.3798219561576843}, 
recall: {'Labels_Average': 0.7787302732467651, 'ECHO': 0.8131868243217468, 'HFPC': 0.6000000238418579, 'BBPC': 0.8309859037399292, 'Whistle': 0.8707482814788818}, 
AUC: {'Labels_Average': 0.9093672633171082, 'ECHO': 0.9538460969924927, 'HFPC': 0.9503985047340393, 'BBPC': 0.8810174465179443, 'Whistle': 0.8522069454193115}, 
exact_match: {'Labels_Average': 0.4912280738353729},
                   Filename Site  ECHO_true  ECHO_pred  ECHO_probs  HFPC_true  \
0  KAM_20

100%|██████████| 5/5 [00:00<00:00, 14.50it/s]

Test on site rdl Epoch: 0 Results - 
loss: 30.768, 
accuracy: {'Labels_Average': 0.9282945990562439, 'ECHO': 0.9224806427955627, 'HFPC': 0.930232584476471, 'BBPC': 0.930232584476471, 'Whistle': 0.930232584476471}, 
f1: {'Labels_Average': 0.8675235509872437, 'ECHO': 0.8571428656578064, 'HFPC': 0.8301886916160583, 'BBPC': 0.8571428656578064, 'Whistle': 0.9256198406219482}, 
precision: {'Labels_Average': 0.8506784439086914, 'ECHO': 0.8571428656578064, 'HFPC': 0.7857142686843872, 'BBPC': 0.8709677457809448, 'Whistle': 0.8888888955116272}, 
recall: {'Labels_Average': 0.886602520942688, 'ECHO': 0.8571428656578064, 'HFPC': 0.8799999952316284, 'BBPC': 0.84375, 'Whistle': 0.9655172228813171}, 
AUC: {'Labels_Average': 0.9668694734573364, 'ECHO': 0.9410334825515747, 'HFPC': 0.9711538553237915, 'BBPC': 0.9732603430747986, 'Whistle': 0.9820300936698914}, 
exact_match: {'Labels_Average': 0.7596899271011353},
                   Filename Site  ECHO_true  ECHO_pred    ECHO_probs  \
0  RDL_20200722_1042

<h2>Training the final model on all the data</h2>

In [ ]:
from training.cross_validation import create_test_fold_indices
from sklearn.model_selection import KFold, train_test_split
from models.utils import aggregate_folds_testing_metrics



labels_df = pd.read_csv("../data/labels/Overlaps_1s.csv")
labels_df["ClipFilenamePt"] = labels_df["ClipFilename"] + ".pt"


label_columns = ["ECHO", "HFPC", "CC", "Whistle"]

data_dir = "../data"
processed_spects_dir = data_dir + "/Full_Dataset/Overlaps_1s_hp_1024_resize/"

results_dir = "./final_results"

labels_df = create_test_fold_indices(labels_df, 5)

In [ ]:

use_quantization = True
        
model_class = load_mobilenet_v3_quant if use_quantization else MobileNetMultilabel

model = model_class(
    pretrained=True,
    n_layers=8,
    num_classes=len(label_columns)
)

train_data, val_data = train_test_split(train_data, test_size=0.2, random_state=42, stratify=train_data['Site'])
test_data = val_data #Doesn't matter here, won't be used anyway

run_name = f"Final_model"
if use_quantization:
    run_name = run_name + "_qat"

run_dir = train_model(
    labels_df,
    label_columns,
    model,
    train_data,
    val_data,
    test_data,
    fold_idx=0,
    processed_spects_dir=processed_spects_dir,
    run_name=run_name,
    results_dir="results/final_model",
    training_config=training_config_default,
    use_quantization=use_quantization,
    save_model=True
)